# Retention Business Analysis

This notebook converts the final Extended XGBoost churn probabilities into an operational retention queue. It evaluates top-decile targeting and combines churn probability with monthly recurring revenue (MRR) to estimate monthly revenue exposure.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PREDICTIONS_PATH = Path("../data/xgboost_test_predictions.csv")
OUTPUT_PATH = Path("../data/high_priority_retention_customers.csv")

df = pd.read_csv(PREDICTIONS_PATH, parse_dates=["observation_date"])

print("Test rows:", len(df))
print("Unique accounts:", df["account_id"].nunique())
print("Observed churners:", int(df["is_churn"].sum()))
print("Overall churn rate:", f"{df['is_churn'].mean():.2%}")
display(df.head())


## Top-decile retention targeting

The retention team is assumed to have capacity to contact the highest-risk 10% of the scored population. The threshold is therefore derived from the out-of-time test predictions rather than chosen from an arbitrary probability cutoff.


In [ ]:
ranked = df.sort_values(
    "xgb_churn_probability",
    ascending=False,
).reset_index(drop=True)

target_count = max(1, int(np.ceil(len(ranked) * 0.10)))
top10 = ranked.head(target_count).copy()

HIGH_RISK_THRESHOLD = top10["xgb_churn_probability"].min()

ranked["retention_priority"] = "Standard"
ranked.loc[ranked.index < target_count, "retention_priority"] = "High Priority"

priority_summary = (
    ranked.groupby("retention_priority")["is_churn"]
    .agg(observations="count", churners="sum", churn_rate="mean")
)

display(priority_summary)
print("Customers targeted:", target_count)
print("Top-decile probability threshold:", HIGH_RISK_THRESHOLD)


In [ ]:
top10 = df[df["retention_priority"] == "High Priority"].copy()

total_churners = int(df["is_churn"].sum())
captured_churners = int(top10["is_churn"].sum())

capture_rate = captured_churners / total_churners
overall_churn_rate = df["is_churn"].mean()
top10_churn_rate = top10["is_churn"].mean()
lift = top10_churn_rate / overall_churn_rate

print("Customers targeted:", len(top10))
print("Churners identified:", captured_churners)
print("Total churners:", total_churners)
print(f"Churners captured: {capture_rate:.2%}")
print(f"Overall churn rate: {overall_churn_rate:.2%}")
print(f"Top-decile churn rate: {top10_churn_rate:.2%}")
print(f"Lift: {lift:.2f}x")


## Expected monthly revenue at risk

For prioritization, churn risk is combined with current MRR:

**expected monthly revenue at risk = predicted churn probability × current MRR**

This is an expected-value ranking measure, not a guaranteed loss forecast.


In [ ]:
df["expected_monthly_revenue_at_risk"] = (
    df["xgb_churn_probability"] * df["current_mrr"]
)

high_risk = (
    df[df["retention_priority"] == "High Priority"]
    .sort_values("expected_monthly_revenue_at_risk", ascending=False)
    .copy()
)

total_revenue_at_risk = df["expected_monthly_revenue_at_risk"].sum()
high_priority_revenue_at_risk = high_risk["expected_monthly_revenue_at_risk"].sum()

print("Expected monthly revenue at risk:", round(total_revenue_at_risk, 2))
print("High-priority expected monthly revenue at risk:", round(high_priority_revenue_at_risk, 2))
print(
    "Share concentrated in high-priority group:",
    f"{high_priority_revenue_at_risk / total_revenue_at_risk:.1%}"
)

display(
    high_risk[
        [
            "account_id",
            "observation_date",
            "current_plan",
            "current_mrr",
            "xgb_churn_probability",
            "expected_monthly_revenue_at_risk",
            "upgrade_last_90d",
            "downsell_last_90d",
            "is_churn",
        ]
    ].head(25)
)


In [ ]:
high_risk.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print("Rows:", len(high_risk))


## Interpretation

The top-decile analysis measures ranking quality and operational concentration: how many actual churners and how much modeled revenue exposure are concentrated inside the segment a retention team could realistically contact. Predictive associations should not be interpreted as causal treatment effects.
